In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [38]:
df = pd.read_csv('D:\\Semester 5\\ML\\244107020017_Machine_Learning\\JS11\\dataset\\voice.csv')

print(df.head())
print(df.shape)
print(df.info())
print(df['label'].value_counts())

   meanfreq        sd    median       Q25       Q75       IQR       skew  \
0  0.059781  0.064241  0.032027  0.015071  0.090193  0.075122  12.863462   
1  0.066009  0.067310  0.040229  0.019414  0.092666  0.073252  22.423285   
2  0.077316  0.083829  0.036718  0.008701  0.131908  0.123207  30.757155   
3  0.151228  0.072111  0.158011  0.096582  0.207955  0.111374   1.232831   
4  0.135120  0.079146  0.124656  0.078720  0.206045  0.127325   1.101174   

          kurt    sp.ent       sfm  ...  centroid   meanfun    minfun  \
0   274.402906  0.893369  0.491918  ...  0.059781  0.084279  0.015702   
1   634.613855  0.892193  0.513724  ...  0.066009  0.107937  0.015826   
2  1024.927705  0.846389  0.478905  ...  0.077316  0.098706  0.015656   
3     4.177296  0.963322  0.727232  ...  0.151228  0.088965  0.017798   
4     4.333713  0.971955  0.783568  ...  0.135120  0.106398  0.016931   

     maxfun   meandom    mindom    maxdom   dfrange   modindx  label  
0  0.275862  0.007812  0.007812  

In [39]:
X = df.drop('label', axis=1)
y = df['label']

print("Jumlah fitur:", X.shape[1])
print("Jumlah data:", X.shape[0])

Jumlah fitur: 20
Jumlah data: 3168


In [40]:
hasil = []

for rasio in [0.3, 0.2]:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=rasio,
        random_state=42,
        stratify=y
    )

    for kernel in ['linear', 'poly', 'rbf']:
        model = make_pipeline(
            StandardScaler(),
            SVC(kernel=kernel)
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        hasil.append({
            'Split': f'{int((1-rasio)*100)}:{int(rasio*100)}',
            'Kernel': kernel,
            'Akurasi': accuracy_score(y_test, y_pred)
        })

df_hasil = pd.DataFrame(hasil)
display(df_hasil.round(4))

,Split,Kernel,Akurasi
0,70:30,linear,0.9790
1,70:30,poly,0.9600
2,70:30,rbf,0.9832
3,80:20,linear,0.9748
4,80:20,poly,0.9558
5,80:20,rbf,0.9826


In [41]:
terbaik = df_hasil.loc[df_hasil['Akurasi'].idxmax()]

print("Split terbaik :", terbaik['Split'])
print("Kernel terbaik:", terbaik['Kernel'])
print(f"Akurasi       : {terbaik['Akurasi']:.4f}")

Split terbaik : 70:30
Kernel terbaik: rbf
Akurasi       : 0.9832


In [42]:
# Membuat fungsi ekstraksi fitur histogram
def extract_hist_feature(img_list, bins=32):
    features = []
    labels = []

    for img, label in img_list:
        img = np.asarray(img)

        if img.max() <= 1:
            hist_range = (0, 1)
        else:
            hist_range = (0, 256)

        hist, _ = np.histogram(
            img.ravel(),
            bins=bins,
            range=hist_range
        )

        hist = hist / hist.sum()
        features.append(hist)
        labels.append(label)

    return np.array(features), np.array(labels)

In [43]:
# Memuat dataset gambar
train_dir = 'dataset/training'
test_dir = 'dataset/test'

train_img_list = load_dataset(train_dir)
test_img_list = load_dataset(test_dir)

print("Jumlah training:", len(train_img_list))
print("Jumlah testing:", len(test_img_list))
print("Label training:", np.unique(
    [label for img, label in train_img_list],
    return_counts=True
))
print("Label testing:", np.unique(
    [label for img, label in test_img_list],
    return_counts=True
))


Jumlah training: 240
Jumlah testing: 160
Label training: (array(['day', 'night'], dtype='<U5'), array([120, 120]))
Label testing: (array(['day', 'night'], dtype='<U5'), array([80, 80]))


In [44]:

# Ekstraksi fitur histogram
X_hist, y_hist = extract_hist_feature(
    train_img_list + test_img_list
)

# Pembagian data 80:20
X_train_hist, X_test_hist, y_train_hist, y_test_hist = train_test_split(
    X_hist, y_hist,
    test_size=0.2,
    random_state=42,
    stratify=y_hist
)

print("Data training:", X_train_hist.shape)
print("Data testing:", X_test_hist.shape)
print("Label:", np.unique(y_hist, return_counts=True))


Data training: (320, 32)
Data testing: (80, 32)
Label: (array(['day', 'night'], dtype='<U5'), array([200, 200]))


In [45]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model_rbf = make_pipeline(
    StandardScaler(),
    SVC(kernel='rbf')
)

model_rbf.fit(X_train_hist, y_train_hist)

y_pred = model_rbf.predict(X_test_hist)

akurasi = accuracy_score(y_test_hist, y_pred)
print(f'Akurasi SVM RBF: {akurasi:.4f}')

Akurasi SVM RBF: 1.0000


In [46]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'svc__C': [0.1, 1, 10, 100],
    'svc__gamma': ['scale', 0.01, 0.1, 1]
}

model_tuning = make_pipeline(
    StandardScaler(),
    SVC(kernel='rbf')
)

grid = GridSearchCV(
    model_tuning,
    param_grid,
    cv=5,
    scoring='accuracy'
)

grid.fit(X_train_hist, y_train_hist)

print("Parameter terbaik:", grid.best_params_)
print(f"Akurasi validasi terbaik: {grid.best_score_:.4f}")

y_pred_tuning = grid.predict(X_test_hist)
akurasi_tuning = accuracy_score(y_test_hist, y_pred_tuning)

print(f"Akurasi test setelah tuning: {akurasi_tuning:.4f}")

Parameter terbaik: {'svc__C': 10, 'svc__gamma': 0.1}
Akurasi validasi terbaik: 0.9906
Akurasi test setelah tuning: 1.0000
